# Contextual Bayesian Optimization：初めてのAsk-and-Tell

このNotebookでは、**操作できる設定値**と**観測するContext**を区別し、候補提案→実験→結果登録→方策評価を順番に実行します。

関連：[理論](../../../docs/theory/29_contextual_bayesian_optimization.md) / [実装ガイド](../../../docs/guide/29_contextual_bayesian_optimization.md)

## 1. 問題設定

- `temperature`：自分で設定できる温度（意思決定）
- `ambient`：実験時に観測する外気条件（Context）
- 品質：`1 - (temperature - ambient)**2`（説明用の模擬目的関数）

目的は単一の温度ではなく、外気条件に応じて温度を変える方策を得ることです。

In [ ]:
from robotorchan.study import create_study

study = create_study(
    search_space={"temperature": (0.0, 1.0), "ambient": (0.0, 1.0)},
    context_names=("ambient",),
    n_initial_trials=3,
    seed=7,
    optimizer_kwargs={"num_restarts": 2, "raw_samples": 16},
)

## 2. Contextを観測して候補を提案

`ask(context=...)`は外気条件を固定して温度を提案します。初期候補を蓄積した後はGPを学習し、獲得関数を最適化します。`tell()`には**実測値**を渡します。

In [ ]:
contexts = (0.2, 0.8, 0.4, 0.6, 0.3)
for ambient in contexts:
    trial = study.ask(context={"ambient": ambient})
    temperature = trial.params["temperature"]
    quality = 1.0 - (temperature - ambient) ** 2
    study.tell(trial, quality)
    print(f"trial={trial.number}, ambient={ambient:.2f}, temperature={temperature:.3f}, quality={quality:.3f}")

## 3. 次のContextに対する候補

学習済みの履歴を使って、新しく観測したContextに対する設定値を提案します。これは**次の実験候補**であり、最適方策の証明ではありません。

In [ ]:
next_trial = study.ask(context={"ambient": 0.7})
print(next_trial.params)
# 実験した場合のみ: study.tell(next_trial, measured_quality)

## 4. 既知の方策の期待値を評価

ここではGPの予測ではなく、あらかじめ与えた2つの方策の目的値を比較します。

In [ ]:
import torch
from robotorchan.optim.contextual_policy import expected_policy_value

contexts_eval = torch.tensor([0.2, 0.5, 0.8], dtype=torch.double)
fixed = torch.full_like(contexts_eval, 0.5)
ideal = contexts_eval
outcomes = torch.stack((1 - (fixed - contexts_eval).square(), 1 - (ideal - contexts_eval).square()))
weights = torch.ones_like(contexts_eval)
print("固定設定と理論最適方策の期待値:", expected_policy_value(outcomes, weights))

## 5. 解釈と次のステップ

理論最適方策 `temperature=ambient` は目的関数を既知として導いたものです。**学習したStudyの未知Contextにおける性能ではありません**。学習済み手法の比較は `benchmarks/contextual_study.py` を参照してください。異なるContextを同時に渡す `ask_contexts` は各Context独立のq=1提案であり、joint q-batchではありません。